In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder

import pickle

In [4]:
data=pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [5]:
data=data.drop(['RowNumber','CustomerId','Surname'],axis=1)

In [6]:
#encode the categorical data
label_encoder_gender=LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

In [11]:
#one-hot encode the geography
onehot_encoder_geography=OneHotEncoder(handle_unknown='ignore')
geo_encoded = onehot_encoder_geography.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geography.get_feature_names_out(['Geography']))
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [12]:
#combine one-hot encoded geography with the original data
data = pd.concat([data, geo_encoded_df], axis=1)
data = data.drop(['Geography'], axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [16]:
#split the data into features and target
X = data.drop(['EstimatedSalary'], axis=1)
y = data['EstimatedSalary']

In [17]:
## Split the data in training and tetsing sets
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [18]:
## load the encoder and scaler
with open('onehot_encoder_geo.pkl','rb') as file:
    label_encoder_geo=pickle.load(file)

with open('label_encoder_gender.pkl', 'rb') as file:
    label_encoder_gender = pickle.load(file)

with open('scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)

## ANN Regression Problem Statement

In [21]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [31]:
#Build the model
model=Sequential([
    Dense(64,activation='relu',input_shape=(X_train.shape[1],)), ## HL1 Connected wwith input layer
    Dense(32,activation='relu'), ## HL2
    Dense(1)  ## output layer for regression
])

## Compile the model
model.compile(optimizer='adam',loss='mean_absolute_error',metrics=['mae'])

model.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_6 (Dense)             (None, 64)                832       
                                                                 
 dense_7 (Dense)             (None, 32)                2080      
                                                                 
 dense_8 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [32]:
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

#set up the Tensorboard
log_dir = "regressionlogs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [33]:
#setup the early stopping
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [34]:
# Train the model
history=model.fit(
    X_train,y_train,validation_data=(X_test,y_test),epochs=100,
    callbacks=[early_stopping_callback, tensorboard_callback]
)

Epoch 1/100
250/250 [==============================] - 2s 5ms/step - loss: 70926.3594 - mae: 70926.3594 - val_loss: 67627.3203 - val_mae: 67627.3203
Epoch 2/100
250/250 [==============================] - 1s 5ms/step - loss: 68220.5078 - mae: 68220.5078 - val_loss: 66158.0547 - val_mae: 66158.0547
Epoch 3/100
250/250 [==============================] - 1s 4ms/step - loss: 65493.6875 - mae: 65493.6875 - val_loss: 61992.3516 - val_mae: 61992.3516
Epoch 4/100
250/250 [==============================] - 1s 5ms/step - loss: 60154.2148 - mae: 60154.2148 - val_loss: 55918.5703 - val_mae: 55918.5703
Epoch 5/100
250/250 [==============================] - 1s 4ms/step - loss: 53948.6406 - mae: 53948.6406 - val_loss: 51546.2656 - val_mae: 51546.2656
Epoch 6/100
250/250 [==============================] - 1s 4ms/step - loss: 51541.6680 - mae: 51541.6680 - val_loss: 51363.9375 - val_mae: 51363.9375
Epoch 7/100
250/250 [==============================] - 1s 4ms/step - loss: 50945.3477 - mae: 50945.3477 - 

In [36]:
%load_ext tensorboard
%tensorboard --logdir regressionlogs/fit

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 26064), started 0:00:11 ago. (Use '!kill 26064' to kill it.)

In [39]:
## Evaluate model on the test data
test_loss, test_mae=model.evaluate(X_test,y_test)
print(f'Test Mae : {test_mae}')

63/63 [==============================] - 0s 3ms/step - loss: 50665.6719 - mae: 50665.6719
Test Mae : 50665.671875
